In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
load_dotenv()

True

In [5]:
llm=ChatOpenAI()

In [6]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [7]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [8]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [9]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [10]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'Football'}, config=config1)

{'topic': 'Football',
 'joke': 'Why did the football coach go to the bank?\n\nTo get his quarterback!',
 'explanation': 'This joke is a play on words. The term "quarterback" is used in football to refer to the player who is responsible for leading the team\'s offensive plays. In this joke, the football coach goes to the bank not to get money, but to "get" his quarterback, meaning to physically retrieve or collect the player. The joke is humorous because it subverts the expectation of why someone would go to a bank.'}

In [11]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football coach go to the bank?\n\nTo get his quarterback!', 'explanation': 'This joke is a play on words. The term "quarterback" is used in football to refer to the player who is responsible for leading the team\'s offensive plays. In this joke, the football coach goes to the bank not to get money, but to "get" his quarterback, meaning to physically retrieve or collect the player. The joke is humorous because it subverts the expectation of why someone would go to a bank.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a8bbf-466c-6014-8002-a5c53d36cb6e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-04T23:54:23.792084+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a8bbf-331f-629a-8001-e1078128354d'}}, tasks=(), interrupts=())

In [12]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football coach go to the bank?\n\nTo get his quarterback!', 'explanation': 'This joke is a play on words. The term "quarterback" is used in football to refer to the player who is responsible for leading the team\'s offensive plays. In this joke, the football coach goes to the bank not to get money, but to "get" his quarterback, meaning to physically retrieve or collect the player. The joke is humorous because it subverts the expectation of why someone would go to a bank.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a8bbf-466c-6014-8002-a5c53d36cb6e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-04T23:54:23.792084+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a8bbf-331f-629a-8001-e1078128354d'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the f

In [13]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'Cricket'}, config=config2)

{'topic': 'Cricket',
 'joke': 'Why did the cricket team go to the bakery?\nBecause they heard they could get a good pitch there!',
 'explanation': 'This joke plays on the multiple meanings of the word "pitch." In cricket, a pitch is the playing area where the match takes place. However, in a bakery, pitch can also refer to the kind of surface on which bread is baked. So, the joke suggests that the cricket team went to the bakery to find a good playing surface, but instead found a good surface for baking bread.'}

In [14]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team go to the bakery?\nBecause they heard they could get a good pitch there!', 'explanation': 'This joke plays on the multiple meanings of the word "pitch." In cricket, a pitch is the playing area where the match takes place. However, in a bakery, pitch can also refer to the kind of surface on which bread is baked. So, the joke suggests that the cricket team went to the bakery to find a good playing surface, but instead found a good surface for baking bread.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a8cba-ac76-6e26-8002-ce0ea1bbfb91'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-05T01:46:52.221997+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a8cba-9a12-6d02-8001-54f959ff6662'}}, tasks=(), interrupts=())

In [15]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team go to the bakery?\nBecause they heard they could get a good pitch there!', 'explanation': 'This joke plays on the multiple meanings of the word "pitch." In cricket, a pitch is the playing area where the match takes place. However, in a bakery, pitch can also refer to the kind of surface on which bread is baked. So, the joke suggests that the cricket team went to the bakery to find a good playing surface, but instead found a good surface for baking bread.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a8cba-ac76-6e26-8002-ce0ea1bbfb91'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-05T01:46:52.221997+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a8cba-9a12-6d02-8001-54f959ff6662'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket

## Time Travel

We use it maybe incase of debugging

In [16]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f1a8cba-ac76-6e26-8002-ce0ea1bbfb91"}})

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team go to the bakery?\nBecause they heard they could get a good pitch there!', 'explanation': 'This joke plays on the multiple meanings of the word "pitch." In cricket, a pitch is the playing area where the match takes place. However, in a bakery, pitch can also refer to the kind of surface on which bread is baked. So, the joke suggests that the cricket team went to the bakery to find a good playing surface, but instead found a good surface for baking bread.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f1a8cba-ac76-6e26-8002-ce0ea1bbfb91'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-05T01:46:52.221997+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a8cba-9a12-6d02-8001-54f959ff6662'}}, tasks=(), interrupts=())

In [18]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f1a8cba-ac76-6e26-8002-ce0ea1bbfb91"}})

{'topic': 'Cricket',
 'joke': 'Why did the cricket team go to the bakery?\nBecause they heard they could get a good pitch there!',
 'explanation': 'This joke plays on the multiple meanings of the word "pitch." In cricket, a pitch is the playing area where the match takes place. However, in a bakery, pitch can also refer to the kind of surface on which bread is baked. So, the joke suggests that the cricket team went to the bakery to find a good playing surface, but instead found a good surface for baking bread.'}

In [19]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team go to the bakery?\nBecause they heard they could get a good pitch there!', 'explanation': 'This joke plays on the multiple meanings of the word "pitch." In cricket, a pitch is the playing area where the match takes place. However, in a bakery, pitch can also refer to the kind of surface on which bread is baked. So, the joke suggests that the cricket team went to the bakery to find a good playing surface, but instead found a good surface for baking bread.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a8cbf-8db7-6b5c-8003-05e00ac1ace6'}}, metadata={'source': 'fork', 'step': 3, 'parents': {}}, created_at='2026-09-05T01:49:03.215633+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a8cba-ac76-6e26-8002-ce0ea1bbfb91'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket

### Update State

In [23]:
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f1a8cba-ac76-6e26-8002-ce0ea1bbfb91", "checkpoint_ns": ""}}, {'topic':'Tennis'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1a8cce-f843-6084-8003-392b9e31deb6'}}

In [24]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Tennis', 'joke': 'Why did the cricket team go to the bakery?\nBecause they heard they could get a good pitch there!', 'explanation': 'This joke plays on the multiple meanings of the word "pitch." In cricket, a pitch is the playing area where the match takes place. However, in a bakery, pitch can also refer to the kind of surface on which bread is baked. So, the joke suggests that the cricket team went to the bakery to find a good playing surface, but instead found a good surface for baking bread.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a8cce-f843-6084-8003-392b9e31deb6'}}, metadata={'source': 'update', 'step': 3, 'parents': {}}, created_at='2026-09-05T01:55:57.040833+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a8cba-ac76-6e26-8002-ce0ea1bbfb91'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricke